# eps_rel damping at full polar — local-basin comparison (opc r=256 + r=64, OLMo + Llama)

**Stream A.** At full polar, does one fixed relative-damping value flatten the local lr basin (lr × {1/3,1,3} around each model's own full-polar optimum) on the weak model (OLMo) **without hurting** Llama? Scope: full-polar arms (ns≥8 / polar-express) + the ns=5 partial-polar reference + AdamW; SSC-clip variants excluded. Labels/colors come from the shared `canonical_label` / `canonical_colors` (AdamW black + first; every axis explicit, so no silent merge — the figure's `assert_label_discriminates` guard hard-errors otherwise). `allow_partial=True`. σ proxy 0.0017.

Same comparison run at **r=256** (top two cells) and **r=64** (bottom two). The r=64 eps_rel arms mirror the r=256 campaign (ns8), with lrs centered on r=64's own full-polar optimum (OLMo k1≈1e-2 / k2≈3e-2; Llama ≈3e-3).

NOTE: OLMo's full-abs reference is `polar-express` (no ns=8-abs run exists on OLMo); Llama's is `ns=8` — labeled honestly by `canonical_label`, not folded. polar-express is a full-polar reference exactly like ns≥8 — `damping_scope` folds both as full.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from lora_playground.plotting import leaderboard_panel, canonical_label

# Run membership comes from the shared registry (lora_playground.workloads) — the
# SAME source the leaderboard doc uses, so notebook and doc cannot drift. Cells are
# leaderboard_panel(model, dataset, rank, ...) + an optional label_filter(label, cfg).

def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def is_curv(label):
    # curvature-whitening / SOAP-curv / KL-Shampoo arms.
    return ('SOAP-curv' in label) or ('KL-Shampoo' in label) or ('+curv' in label)

OPT_CT = 'adam-polar-product-lora-coupled-spectral-chord-tight'

def damp_keep(label, cfg):
    """Full-polar (ns>=8 / polar_express) damping comparison: abs vs eps_rel, plus
    the plain ns=5 partial-polar reference. Excludes the SSC-clip arms."""
    if label == 'AdamW':
        return True
    if cfg.get('ssc_kappa') is not None or cfg.get('ssc_c') is not None:
        return False
    ns = ns_of(cfg)
    full = (cfg.get('polar_method') == 'polar_express') or (ns is not None and ns >= 8)
    if full:
        return True
    if cfg.get('optimizer') == OPT_CT and ns == 5 and not cfg.get('precond_delta_relative'):
        return True
    return False

## OLMo-2-1B × opc × r=256 — the weak/blowup model (full-polar optimum ≈ 1e-2)

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('allenai/OLMo-2-0425-1B', 'opc', 256,
    'OLMo-2-1B × opc × r=256 — eps_rel damping at full polar',
    label_filter=damp_keep,
    figsize=(13, 5))
plt.show()
sdf

## Llama-3.2-1B × opc × r=256 — the good model (full-polar optimum ≈ 1e-3); does eps_rel hurt?

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 256,
    'Llama-3.2-1B × opc × r=256 — eps_rel damping at full polar',
    label_filter=damp_keep,
    figsize=(13, 5))
plt.show()
sdf

## OLMo-2-1B × opc × r=64 — full-polar optimum ≈ 1e-2 (k1) / 3e-2 (k2)

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('allenai/OLMo-2-0425-1B', 'opc', 64,
    'OLMo-2-1B × opc × r=64 — eps_rel damping at full polar',
    label_filter=damp_keep,
    figsize=(13, 5))
plt.show()
sdf

## Llama-3.2-1B × opc × r=64 — full-polar optimum ≈ 3e-3; does eps_rel hurt?

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 64,
    'Llama-3.2-1B × opc × r=64 — eps_rel damping at full polar',
    label_filter=damp_keep,
    figsize=(13, 5))
plt.show()
sdf